In [ ]:
# =========================
# nb_validate
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def load_rules_config():
    # optional config override
    path = "/lakehouse/default/Files/config/rules_config.json"
    try:
        raw = mssparkutils.fs.head(path, 2_000_000)
        import json
        cfg = json.loads(raw)
        return cfg
    except Exception:
        return {}

def is_rule_enabled(cfg, rule_id, default=True):
    # supports both:
    # {"MV_001":{"enabled":true}} or {"rules":{"MV_001":{"enabled":true}}}
    if "rules" in cfg and isinstance(cfg["rules"], dict):
        r = cfg["rules"].get(rule_id, {})
        return r.get("enabled", default)
    r = cfg.get(rule_id, {})
    return r.get("enabled", default) if isinstance(r, dict) else default

def rule_param(cfg, rule_id, key, default):
    if "rules" in cfg and isinstance(cfg["rules"], dict):
        return cfg["rules"].get(rule_id, {}).get(key, default)
    return cfg.get(rule_id, {}).get(key, default) if isinstance(cfg.get(rule_id, {}), dict) else default

# ---------- Load canonical for run ----------
ch = spark.table("canonical_holdings").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if ch.rdd.isEmpty():
    print(f"[nb_validate] No canonical_holdings rows for period={period}, run_id={run_id}")
    mssparkutils.notebook.exit("NO_DATA")

# ---------- Rule config ----------
cfg = load_rules_config()
mv_enabled   = is_rule_enabled(cfg, "MV_001", True)
date_enabled = is_rule_enabled(cfg, "DATE_001", True)
val_enabled  = is_rule_enabled(cfg, "VAL_001", True)
map_enabled  = is_rule_enabled(cfg, "MAP_001", True)

mv_tol = float(rule_param(cfg, "MV_001", "tolerance_pct", 0.02))  # ±2% default
stale_days = int(rule_param(cfg, "DATE_001", "max_age_days", 35))  # PoC weekend-only approximation

# ---------- Common projection helper ----------
def to_event(df, rule_id, severity, status, details_expr):
    return (
        df.select(
            F.col("period"),
            F.col("run_id"),
            F.current_timestamp().alias("event_time"),
            F.col("dfm_id"),
            F.col("dfm_name"),
            F.col("policy_id"),
            F.col("security_id"),
            F.lit(rule_id).alias("rule_id"),
            F.lit(severity).alias("severity"),
            F.lit(status).alias("status"),
            details_expr.alias("details_json"),
            F.col("source_file")
        )
    )

events = None

# ============================================================
# MV_001: Market value check
# Evaluability: local_bid_price, holding, bid_value_gbp, fx_rate must be present and fx_rate != 0
# expected_gbp = holding * local_bid_price * fx_rate
# fail if abs(actual-expected) / abs(expected) > tolerance
# ============================================================
if mv_enabled:
    mv_base = ch.withColumn(
        "expected_bid_value_gbp",
        F.col("holding").cast("double") * F.col("local_bid_price").cast("double") * F.col("fx_rate").cast("double")
    ).withColumn(
        "actual_bid_value_gbp",
        F.col("bid_value_gbp").cast("double")
    )

    mv_not_eval = mv_base.filter(
        F.col("holding").isNull() |
        F.col("local_bid_price").isNull() |
        F.col("fx_rate").isNull() |
        (F.col("fx_rate") == 0) |
        F.col("actual_bid_value_gbp").isNull()
    )

    mv_not_eval_ev = to_event(
        mv_not_eval,
        "MV_001",
        "warning",
        "not_evaluable",
        F.to_json(F.struct(
            F.lit("missing_inputs").alias("reason"),
            F.col("holding"),
            F.col("local_bid_price"),
            F.col("fx_rate"),
            F.col("bid_value_gbp")
        ))
    )

    mv_eval = mv_base.filter(
        F.col("holding").isNotNull() &
        F.col("local_bid_price").isNotNull() &
        F.col("fx_rate").isNotNull() &
        (F.col("fx_rate") != 0) &
        F.col("actual_bid_value_gbp").isNotNull() &
        F.col("expected_bid_value_gbp").isNotNull() &
        (F.abs(F.col("expected_bid_value_gbp")) > 0)
    ).withColumn(
        "pct_diff",
        F.abs(F.col("actual_bid_value_gbp") - F.col("expected_bid_value_gbp")) / F.abs(F.col("expected_bid_value_gbp"))
    )

    mv_fail = mv_eval.filter(F.col("pct_diff") > mv_tol)

    mv_fail_ev = to_event(
        mv_fail,
        "MV_001",
        "exception",
        "fail",
        F.to_json(F.struct(
            F.col("actual_bid_value_gbp").alias("actual"),
            F.col("expected_bid_value_gbp").alias("expected"),
            F.col("pct_diff"),
            F.lit(mv_tol).alias("tolerance_pct")
        ))
    )

    events = mv_not_eval_ev.unionByName(mv_fail_ev) if events is None else events.unionByName(mv_not_eval_ev).unionByName(mv_fail_ev)

# ============================================================
# DATE_001: Date validity/staleness
# fail if report_date null or report_date too old for run period
# ============================================================
if date_enabled:
    # Period end date from period string YYYY-MM
    period_end = F.last_day(F.to_date(F.concat(F.lit(period), F.lit("-01"))))

    date_check = ch.withColumn("period_end", period_end).withColumn(
        "age_days",
        F.datediff(F.col("period_end"), F.col("report_date"))
    )

    date_fail = date_check.filter(
        F.col("report_date").isNull() |
        (F.col("age_days") > stale_days) |
        (F.col("age_days") < -5)  # obvious future-date anomaly guard
    )

    date_fail_ev = to_event(
        date_fail,
        "DATE_001",
        "warning",
        "fail",
        F.to_json(F.struct(
            F.col("report_date"),
            F.col("period_end"),
            F.col("age_days"),
            F.lit(stale_days).alias("max_age_days")
        ))
    )

    events = date_fail_ev if events is None else events.unionByName(date_fail_ev)

# ============================================================
# VAL_001: Policy-level aggregate sanity
# fail when total_bid_value_gbp <= 0 for a policy in run
# ============================================================
if val_enabled:
    pol = (
        ch.groupBy("period", "run_id", "dfm_id", "dfm_name", "policy_id")
        .agg(
            F.sum(F.coalesce(F.col("bid_value_gbp").cast("double"), F.lit(0.0))).alias("total_bid_value_gbp"),
            F.count("*").alias("row_count")
        )
    )

    val_fail = pol.filter(F.col("total_bid_value_gbp") <= 0)

    # attach source_file/security_id as null for policy-level events
    val_fail_ev = (
        val_fail
        .withColumn("security_id", F.lit(None).cast("string"))
        .withColumn("source_file", F.lit(None).cast("string"))
        .select(
            "period","run_id",
            F.current_timestamp().alias("event_time"),
            "dfm_id","dfm_name","policy_id","security_id",
            F.lit("VAL_001").alias("rule_id"),
            F.lit("warning").alias("severity"),
            F.lit("fail").alias("status"),
            F.to_json(F.struct("total_bid_value_gbp","row_count")).alias("details_json"),
            "source_file"
        )
    )

    events = val_fail_ev if events is None else events.unionByName(val_fail_ev)

# ============================================================
# MAP_001: Mapping completeness
# fail if policy_id missing OR both security_id and isin missing
# ============================================================
if map_enabled:
    map_fail = ch.filter(
        F.col("policy_id").isNull() |
        (F.col("security_id").isNull() & F.col("isin").isNull())
    )

    map_fail_ev = to_event(
        map_fail,
        "MAP_001",
        "exception",
        "fail",
        F.to_json(F.struct(
            F.col("policy_id"),
            F.col("security_id"),
            F.col("isin")
        ))
    )

    events = map_fail_ev if events is None else events.unionByName(map_fail_ev)

# ---------- Write validation_events ----------
if events is None or events.rdd.isEmpty():
    print("[nb_validate] No validation failures/not_evaluable to write.")
    mssparkutils.notebook.exit("OK")

events.write.mode("append").saveAsTable("validation_events")

# ---------- Summary ----------
summary = (
    events.groupBy("rule_id", "status")
    .count()
    .orderBy("rule_id", "status")
)
display(summary)

print(f"[nb_validate] Completed for period={period}, run_id={run_id}")
mssparkutils.notebook.exit("OK")